In [5]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

import pandas as pd
import pickle
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

# Set paths
oai_data_dir = Path("../../../data/processed/oai")
output_dir = Path("../../../data/processed/traffic")
output_dir.mkdir(parents=True, exist_ok=True)

print("OAI Dataset Conversion Script")
print("=" * 50)
print("Merging train-test pairs into combined pickle files")


OAI Dataset Conversion Script
Merging train-test pairs into combined pickle files


In [6]:
# Find all CSV files and group by flow type (haptic, motion)
csv_files = list(oai_data_dir.glob("*.csv"))

print(f"\nFound {len(csv_files)} CSV files:")
for csv_file in csv_files:
    print(f"  - {csv_file.name}")

# Group files by flow type (e.g., haptic_1ms_20, motion_1ms_20)
flow_groups = {}
for csv_file in csv_files:
    # Extract base name (e.g., "haptic_1ms_20" from "haptic_1ms_20_train.csv")
    parts = csv_file.stem.split('_')
    if len(parts) >= 4 and parts[-1] in ['train', 'test']:
        flow_name = '_'.join(parts[:-1])  # e.g., "haptic_1ms_20"
        split_type = parts[-1]  # "train" or "test"
        
        if flow_name not in flow_groups:
            flow_groups[flow_name] = {}
        flow_groups[flow_name][split_type] = csv_file

print(f"\nGrouped into {len(flow_groups)} flow types:")
for flow_name, files in flow_groups.items():
    print(f"  {flow_name}: {list(files.keys())}")



Found 4 CSV files:
  - haptic_1ms_20_test.csv
  - haptic_1ms_20_train.csv
  - motion_1ms_20_test.csv
  - motion_1ms_20_train.csv

Grouped into 2 flow types:
  haptic_1ms_20: ['test', 'train']
  motion_1ms_20: ['test', 'train']


In [7]:
# Process each flow group (train-test pairs)
print("\nProcessing flow groups...")
print("=" * 50)

for flow_name, files in flow_groups.items():
    print(f"\n{flow_name}:")
    
    # Check if both train and test exist
    if 'train' not in files or 'test' not in files:
        print(f"  ⚠ Skipping - missing train or test file")
        continue
    
    # Read train data
    train_df = pd.read_csv(files['train'])
    print(f"  Train: {train_df.shape} - {files['train'].name}")
    
    # Read test data
    test_df = pd.read_csv(files['test'])
    print(f"  Test:  {test_df.shape} - {files['test'].name}")
    
    # Create combined data structure following EnvironmentSim.py format
    # trainData['actual'], testData['actual'], trainData['predicted'], testData['predicted']
    train_data = {
        'actual': train_df['Traffic_Received'].values,
        'predicted': train_df['Traffic_Predicted'].values
    }
    
    test_data = {
        'actual': test_df['Traffic_Received'].values,
        'predicted': test_df['Traffic_Predicted'].values
    }

    
    # Save train predictions
    train_output = output_dir / f"{flow_name}_train.pkl"
    with open(train_output, 'wb') as f:
        pickle.dump(train_data, f)
    print(f"  ✓ Saved: {train_output.name} ({train_output.stat().st_size / 1024:.2f} KB)")
    
    # Save test predictions
    test_output = output_dir / f"{flow_name}_test.pkl"
    with open(test_output, 'wb') as f:
        pickle.dump(test_data, f)
    print(f"  ✓ Saved: {test_output.name} ({test_output.stat().st_size / 1024:.2f} KB)")

print("\n" + "=" * 50)
print("Conversion complete!")



Processing flow groups...

haptic_1ms_20:
  Train: (11610, 3) - haptic_1ms_20_train.csv
  Test:  (7486, 3) - haptic_1ms_20_test.csv
  ✓ Saved: haptic_1ms_20_train.pkl (181.63 KB)
  ✓ Saved: haptic_1ms_20_test.pkl (117.17 KB)

motion_1ms_20:
  Train: (18955, 3) - motion_1ms_20_train.csv
  Test:  (7762, 3) - motion_1ms_20_test.csv
  ✓ Saved: motion_1ms_20_train.pkl (296.39 KB)
  ✓ Saved: motion_1ms_20_test.pkl (121.49 KB)

Conversion complete!


In [9]:
# Verify saved pickle files
print("\nVerifying saved pickle files...")
print("=" * 50)

for flow_name in flow_groups.keys():
    print(f"\n{flow_name}:")
    
    # Verify train file
    train_file = output_dir / f"{flow_name}_train.pkl"
    if train_file.exists():
        with open(train_file, 'rb') as f:
            train_data = pickle.load(f)
        print(f"  ✓ Train predictions:")
        print(f"      Keys: {list(train_data.keys())}")
        print(f"      actual shape: {train_data['actual'].shape}")
        print(f"      predicted shape: {train_data['predicted'].shape}")
    else:
        print(f"  ✗ Train file not found")
    
    # Verify test file
    test_file = output_dir / f"{flow_name}_test.pkl"
    if test_file.exists():
        with open(test_file, 'rb') as f:
            test_data = pickle.load(f)
        print(f"  ✓ Test predictions:")
        print(f"      Keys: {list(test_data.keys())}")
        print(f"      actual shape: {test_data['actual'].shape}")
        print(f"      predicted shape: {test_data['predicted'].shape}")
    else:
        print(f"  ✗ Test file not found")

print("\n" + "=" * 50)
print("Verification complete!")



Verifying saved pickle files...

haptic_1ms_20:
  ✓ Train predictions:
      Keys: ['actual', 'predicted']
      actual shape: (11610,)
      predicted shape: (11610,)
  ✓ Test predictions:
      Keys: ['actual', 'predicted']
      actual shape: (7486,)
      predicted shape: (7486,)

motion_1ms_20:
  ✓ Train predictions:
      Keys: ['actual', 'predicted']
      actual shape: (18955,)
      predicted shape: (18955,)
  ✓ Test predictions:
      Keys: ['actual', 'predicted']
      actual shape: (7762,)
      predicted shape: (7762,)

Verification complete!


In [10]:
# Display statistics for the converted datasets
print("\nDataset Statistics")
print("=" * 50)

for flow_name in flow_groups.keys():
    print(f"\n{flow_name}:")
    
    # Load train data
    train_file = output_dir / f"{flow_name}_train.pkl"
    with open(train_file, 'rb') as f:
        train_data = pickle.load(f)
    
    # Load test data
    test_file = output_dir / f"{flow_name}_test.pkl"
    with open(test_file, 'rb') as f:
        test_data = pickle.load(f)
    
    print(f"  Train:")
    print(f"    Samples: {len(train_data['actual'])}")
    print(f"    Actual   - Mean: {np.mean(train_data['actual']):.2f}, Std: {np.std(train_data['actual']):.2f}")
    print(f"    Predicted - Mean: {np.mean(train_data['predicted']):.2f}, Std: {np.std(train_data['predicted']):.2f}")
    
    print(f"  Test:")
    print(f"    Samples: {len(test_data['actual'])}")
    print(f"    Actual   - Mean: {np.mean(test_data['actual']):.2f}, Std: {np.std(test_data['actual']):.2f}")
    print(f"    Predicted - Mean: {np.mean(test_data['predicted']):.2f}, Std: {np.std(test_data['predicted']):.2f}")

print("\n" + "=" * 50)
print("All conversions complete!")
print("\nFormat compatible with EnvironmentSim.py:")
print("  trainData['actual'], testData['actual'],")
print("  trainData['predicted'], testData['predicted']")



Dataset Statistics

haptic_1ms_20:
  Train:
    Samples: 11610
    Actual   - Mean: 1.80, Std: 2.80
    Predicted - Mean: 1.49, Std: 0.93
  Test:
    Samples: 7486
    Actual   - Mean: 1.81, Std: 2.27
    Predicted - Mean: 1.45, Std: 0.96

motion_1ms_20:
  Train:
    Samples: 18955
    Actual   - Mean: 2.74, Std: 4.40
    Predicted - Mean: 1.90, Std: 1.81
  Test:
    Samples: 7762
    Actual   - Mean: 2.89, Std: 5.02
    Predicted - Mean: 1.88, Std: 1.88

All conversions complete!

Format compatible with EnvironmentSim.py:
  trainData['actual'], testData['actual'],
  trainData['predicted'], testData['predicted']
